# Install the Phys 119 tools on open.jupyter.ubc.ca

**Only for open.jupyter.ubc.ca**, the backup Jupyter server for Phys 119. On our usual server, phys119.phas.ubc.ca, these tools are already installed for everyone, and this notebook will refuse to run there.

It installs the two tools the course notebooks use:
* `phys119_functions` 1.4.0, for calculations and plots (`plot_data()`, `manual_fit()`, `autofit()`, ...);
* `data_entry2`, for the spreadsheets you enter data in.

Run the cells **one at a time, in order**. Each one checks something before the next step, and stops with a message if anything is not right. Nothing is installed until Step 3, and only into your own account.

## Step 1: Check that this is open.jupyter.ubc.ca

In [ ]:
# Run me to check which Jupyter server this notebook is running on

import os
import sys

checks = [
    ("not the Phys 119 server (phys119.phas.ubc.ca)", not os.path.exists("/opt/tljh")),
    ("running on UBC's open Jupyter image",
     os.environ.get("JUPYTER_IMAGE", "").startswith("032401129069.dkr.ecr.ca-central-1.amazonaws.com/")),
    ("running on a Kubernetes JupyterHub", bool(os.environ.get("KUBERNETES_SERVICE_HOST"))
                                          and bool(os.environ.get("JUPYTERHUB_USER"))),
    ("the server's own Python (/opt/conda)", os.path.realpath(sys.prefix) == "/opt/conda"),
    ("home folder /home/jovyan", os.path.expanduser("~") == "/home/jovyan"),
    ("Python 3.10 or newer", sys.version_info >= (3, 10)),
]
for label, ok in checks:
    print(("OK   " if ok else "NO   ") + label)

server_ok = all(ok for _, ok in checks)
print()
if server_ok:
    print("This is open.jupyter.ubc.ca. Go on to Step 2.")
else:
    raise RuntimeError("STOP: this does not look like open.jupyter.ubc.ca, so nothing will be installed. "
                       "Only run this notebook there.")

## Step 2: Check what is already installed

In [ ]:
# Run me to check what is already installed (nothing is changed)

import importlib.metadata as metadata
import site

if not globals().get("server_ok"):
    raise RuntimeError("STOP: run Step 1 first, on open.jupyter.ubc.ca.")

def installed(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

# Already part of the server; this notebook does not install or change them.
needed = {"numpy": "1.20", "matplotlib": "3.5", "pandas": "1.3.5", "ipywidgets": "7.6",
          "traitlets": "4.3", "typing_extensions": "4.2"}
problems = []
for name, minimum in needed.items():
    version = installed(name)
    print(f"{name:18} {version or 'MISSING'}  (needs {minimum} or newer)")
    if version is None:
        problems.append(name)

# Installed in Step 3 if missing: what anywidget (for manual_fit) and data_entry2
# need, each with the versions those packages ask for.
extras = {"psygnal": "psygnal", "anywidget": "anywidget",
          "gast": "gast>=0.7.0,<0.8", "py2vega": "py2vega>=0.5", "traittypes": "traittypes>=0.0.6",
          "bqscales": "bqscales>=0.3.3,<0.4", "bqplot": "bqplot>=0.11.6", "ipydatagrid": "ipydatagrid",
          "data-entry2": "data-entry2"}
print()
to_install = []
for name, requirement in extras.items():
    version = installed(name)
    print(f"{name:18} {version or 'not installed: Step 3 will install it for you'}")
    if version is None:
        to_install.append(requirement)

current = installed("phys119-functions")
print(f"{'phys119-functions':18} {current or 'not installed'}")
print("\nYour personal package folder:", site.getusersitepackages())

if problems:
    raise RuntimeError("STOP: this server is missing " + ", ".join(problems)
                       + ". Tell your instructor; do not install anything.")
print("\nEverything needed is here. Go on to Step 3.")

## Step 3: Install (into your own account only)

This installs `phys119_functions` 1.4.0, and anything else Step 2 said was missing (such as `data_entry2` and the spreadsheet and slider tools it and `manual_fit()` use), into your personal package folder. `--user` keeps everything inside your own account, and `--no-deps` stops pip from changing any of the server's own packages, such as numpy, pandas or matplotlib.

In [ ]:
# Run me to install the Phys 119 tools for your account

import subprocess

if not globals().get("server_ok") or "to_install" not in globals():
    raise RuntimeError("STOP: run Steps 1 and 2 first.")

wheel = ("https://github.com/phys119/phys119_functions/releases/download/v1.4.0/"
         "phys119_functions-1.4.0-py3-none-any.whl")
command = [sys.executable, "-m", "pip", "install", "--user", "--no-deps", "--no-cache-dir",
           "--upgrade", wheel] + to_install
print("Running:", " ".join(command[2:]), "\n")
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout[-2000:])
if result.returncode != 0:
    print(result.stderr[-2000:])
    raise RuntimeError("STOP: the install did not work. Show this output to your instructor.")
print("Installed. Now do Step 4 before anything else.")

## Step 4: Restart your server

Restart your whole Jupyter server (not just the kernel), so that it finds the new packages and the parts that draw `manual_fit()`'s sliders and `data_entry2`'s spreadsheets:

1. Click **File > Hub Control Panel**.
2. Click **Stop My Server**, wait for it to stop, then click **Start My Server**.
3. Open this notebook again and run **Step 5** (you do not need to run Steps 1 to 4 again).

## Step 5: Check that it works

In [ ]:
# Run me after restarting your server, to check the install

import site
import subprocess
import sys
import numpy as np

import phys119_functions as pf
from phys119_functions import *
import data_entry2
import anywidget

print("phys119_functions", pf.__version__, "from", pf.__file__)
assert pf.__version__ == "1.4.0", "The version is not 1.4.0: did you restart your server (Step 4)?"
assert pf.__file__.startswith(site.getusersitepackages()), "Not loaded from your personal package folder."
print("data_entry2 loaded from", data_entry2.__file__)
print("anywidget", anywidget.__version__)

# The parts of the spreadsheet and slider tools that run in your browser
listing = subprocess.run([sys.executable, "-m", "jupyter", "labextension", "list"],
                         capture_output=True, text=True)
for extension in ("anywidget", "ipydatagrid", "bqplot"):
    lines = [line.strip() for line in (listing.stdout + listing.stderr).splitlines() if extension in line]
    print(f"{extension:12}", lines[0] if lines else "NOT FOUND")
    assert lines and "enabled" in lines[0].lower() and "ok" in lines[0].lower(), (
        f"{extension} is not ready: did you restart your server (Step 4)?")

x = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y = np.array([2.1, 3.9, 6.2, 7.8, 10.1])
dy = np.array([0.2, 0.2, 0.2, 0.2, 0.2])
autofit(x, y, dy, title="Check: autofit")

You should see each tool listed as enabled, then a plot with a best-fit line and residuals, and a line starting `autofit1:` under it. If so, everything is ready: use `from phys119_functions import *` and `import data_entry2` in your notebooks on this server, as on phys119.phas.ubc.ca.

**If it stops working later:** the server's software is occasionally updated. If a notebook says `No module named 'phys119_functions'` or `No module named 'data_entry2'`, run this notebook again from Step 1.